In [1]:
import subprocess
import sys

# Install required libraries for SciBERT
libraries = [
    'transformers',
    'torch',
    'scikit-learn'
]

print("Installing required libraries...")
for lib in libraries:
    print(f"Installing {lib}...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", lib])

print("✓ All libraries installed")

Installing required libraries...
Installing transformers...
Installing torch...
Installing scikit-learn...
✓ All libraries installed


In [2]:
import json
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Load training data
train_data = []
with open("../scicite/train.jsonl", 'r', encoding='utf-8') as f:
    for line in f:
        example = json.loads(line)
        train_data.append({
            'text': example['string'],
            'label': example['label']
        })

print(f"Training data: {len(train_data)} examples")

# Load dev data
dev_data = []
with open("../scicite/dev.jsonl", 'r', encoding='utf-8') as f:
    for line in f:
        example = json.loads(line)
        dev_data.append({
            'text': example['string'],
            'label': example['label']
        })

print(f"Dev data: {len(dev_data)} examples")

# Convert to dataframe
train_df = pd.DataFrame(train_data)
dev_df = pd.DataFrame(dev_data)

print(f"Train: {train_df.shape[0]} rows, Dev: {dev_df.shape[0]} rows")

# Get the labels
y_train = train_df['label'].values
y_dev = dev_df['label'].values
X_train = train_df['text'].values
X_dev = dev_df['text'].values

print(f"Classes: {np.unique(y_train)}")

Training data: 8243 examples
Dev data: 916 examples
Train: 8243 rows, Dev: 916 rows
Classes: ['background' 'method' 'result']


In [3]:
from transformers import AutoTokenizer, AutoModel
import torch

# Load SciBERT model and tokenizer
print("Loading SciBERT model...")
model_name = "allenai/scibert_scivocab_uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
bert_model = AutoModel.from_pretrained(model_name)

print(f"✓ Model loaded: {model_name}")
print(f"Tokenizer vocabulary size: {tokenizer.vocab_size}")

# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

bert_model.to(device)
print("✓ Model moved to device")

Loading SciBERT model...


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

C:\Users\test\AppData\Roaming\Python\Python310\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\test\.cache\huggingface\hub\models--allenai--scibert_scivocab_uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.txt:   0%|          | 0.00/228k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/442M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Model loaded: allenai/scibert_scivocab_uncased
Tokenizer vocabulary size: 31090
Using device: cpu
✓ Model moved to device


model.safetensors:   0%|          | 0.00/442M [00:00<?, ?B/s]

In [6]:
# Convert each sentence to a vector using SciBERT
print("Creating embeddings for training data...")

def get_embeddings(texts, batch_size=32):
    embeddings = []
    # Convert to list if it's numpy array
    texts = list(texts)
    
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        
        # Tokenize the text
        inputs = tokenizer(batch_texts, padding=True, truncation=True, max_length=512, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        # Get embeddings from the model
        with torch.no_grad():
            outputs = bert_model(**inputs)
        
        # Use the [CLS] token embedding (first token)
        cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        embeddings.extend(cls_embeddings)
        
        if (i + batch_size) % 100 == 0 or i + batch_size >= len(texts):
            print(f"  Processed {min(i + batch_size, len(texts))}/{len(texts)}")
    
    return np.array(embeddings)

# Get embeddings for train and dev
X_train_embeddings = get_embeddings(X_train)
print(f"Train embeddings shape: {X_train_embeddings.shape}")

X_dev_embeddings = get_embeddings(X_dev)
print(f"Dev embeddings shape: {X_dev_embeddings.shape}")

Creating embeddings for training data...
  Processed 800/8243
  Processed 1600/8243
  Processed 2400/8243
  Processed 3200/8243
  Processed 4000/8243
  Processed 4800/8243
  Processed 5600/8243
  Processed 6400/8243
  Processed 7200/8243
  Processed 8000/8243
  Processed 8243/8243
Train embeddings shape: (8243, 768)
  Processed 800/916
  Processed 916/916
Dev embeddings shape: (916, 768)


In [7]:
from sklearn.linear_model import LogisticRegression

# Train a simple classifier on top of SciBERT embeddings
print("Training Logistic Regression on SciBERT embeddings...")
classifier = LogisticRegression(max_iter=1000, random_state=42, multi_class='multinomial')
classifier.fit(X_train_embeddings, y_train)
print("✓ Model trained!")

# Make predictions
y_train_pred = classifier.predict(X_train_embeddings)
y_dev_pred = classifier.predict(X_dev_embeddings)

# Calculate metrics
train_accuracy = accuracy_score(y_train, y_train_pred)
train_f1 = f1_score(y_train, y_train_pred, average='macro')
dev_accuracy = accuracy_score(y_dev, y_dev_pred)
dev_f1 = f1_score(y_dev, y_dev_pred, average='macro')

print(f"\nTraining Results:")
print(f"  Accuracy: {train_accuracy:.4f}")
print(f"  Macro F1: {train_f1:.4f}")

print(f"\nDev Results:")
print(f"  Accuracy: {dev_accuracy:.4f}")
print(f"  Macro F1: {dev_f1:.4f}")

Training Logistic Regression on SciBERT embeddings...


C:\Users\test\AppData\Roaming\Python\Python310\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


✓ Model trained!

Training Results:
  Accuracy: 0.8957
  Macro F1: 0.8864

Dev Results:
  Accuracy: 0.8242
  Macro F1: 0.7973


In [8]:
# Show detailed results
print("Detailed Classification Report (SciBERT):")
print(classification_report(y_dev, y_dev_pred))

# Save results
results_scibert = {
    'model': 'SciBERT + Logistic Regression',
    'dev_accuracy': dev_accuracy,
    'dev_f1': dev_f1,
    'train_accuracy': train_accuracy,
    'train_f1': train_f1,
    'classification_report': classification_report(y_dev, y_dev_pred, output_dict=True)
}

import json
with open('../results_scibert.json', 'w') as f:
    json.dump(results_scibert, f, indent=2)

print("\n✓ Results saved to results_scibert.json")

Detailed Classification Report (SciBERT):
              precision    recall  f1-score   support

  background       0.83      0.89      0.86       538
      method       0.81      0.73      0.77       255
      result       0.81      0.72      0.76       123

    accuracy                           0.82       916
   macro avg       0.82      0.78      0.80       916
weighted avg       0.82      0.82      0.82       916


✓ Results saved to results_scibert.json
